<a href="https://colab.research.google.com/github/Johnogunlola/MRes-AI/blob/MRes/Comprehensive_Solution_for_Predictive_Maintenance1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (LSTM, GRU, Dense, Dropout, Input,
                                     MultiHeadAttention, LayerNormalization,
                                     Conv1D, MaxPooling1D, Flatten,
                                     TimeDistributed, concatenate,
                                     GlobalAveragePooling1D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import shap
import joblib

In [ ]:
np.random.seed(42)

In [ ]:
def load_and_preprocess(file_path):
    """Load and preprocess dataset"""
    df = pd.read_csv(file_path)

    # Feature engineering
    df['s21_diff'] = df.groupby('id')['s21'].diff().fillna(0)
    df['s15_rolling'] = df.groupby('id')['s15'].rolling(window=5).mean().reset_index(0, drop=True)
    df['sensor_ratio'] = df['s3'] / (df['s4'] + 1e-6)
    df['op_regime'] = pd.cut(df['setting1'], bins=3, labels=['low', 'medium', 'high'])

    # One-hot encode operational regime
    df = pd.get_dummies(df, columns=['op_regime'], prefix='op')

    # Create RUL target (time to failure)
    df['RUL'] = df.groupby('id')['cycle'].transform('max') - df['cycle']

    # Select relevant features
    feature_cols = [
        'setting1', 'setting2', 's2', 's3', 's4', 's7', 's8', 's11',
        's12', 's15', 's21', 's21_diff', 's15_rolling', 'sensor_ratio',
        'op_low', 'op_medium', 'op_high'
    ]

    return df, feature_cols

In [ ]:
# Feature engineering - Moved to load_and_preprocess function
# df['s21_diff'] = df.groupby('id')['s21'].diff().fillna(0)
# df['s15_rolling'] = df.groupby('id')['s15'].rolling(window=5).mean().reset_index(0, drop=True)
# df['sensor_ratio'] = df['s3'] / (df['s4'] + 1e-6)
# df['op_regime'] = pd.cut(df['setting1'], bins=3, labels=['low', 'medium', 'high'])

In [ ]:
def create_sequences(data, feature_cols, sequence_length=30, stride=5):
    """Create time-series sequences for LSTM/GRU models"""
    sequences = []
    targets = []
    engine_ids = []

    for engine_id in data['id'].unique():
        engine_data = data[data['id'] == engine_id]
        engine_features = engine_data[feature_cols].values
        engine_rul = engine_data['RUL'].values

        for i in range(0, len(engine_data) - sequence_length, stride):
            sequences.append(engine_features[i:i+sequence_length])
            targets.append(engine_rul[i+sequence_length-1])
            engine_ids.append(engine_id)

    return np.array(sequences), np.array(targets), np.array(engine_ids)

In [ ]:
def prepare_data(train_path, test_path):
    """Prepare train and test datasets"""
    # Load and preprocess
    train_df, feature_cols = load_and_preprocess(train_path)
    test_df, _ = load_and_preprocess(test_path)

    # Check for NaNs after preprocessing
    if train_df[feature_cols].isnull().sum().sum() > 0:
        print("NaNs found in training features after preprocessing:")
        print(train_df[feature_cols].isnull().sum()[train_df[feature_cols].isnull().sum() > 0])
    if test_df[feature_cols].isnull().sum().sum() > 0:
        print("NaNs found in testing features after preprocessing:")
        print(test_df[feature_cols].isnull().sum()[test_df[feature_cols].isnull().sum() > 0])


    # Scale features
    scaler = RobustScaler()
    train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    test_df[feature_cols] = scaler.transform(test_df[feature_cols])

    # Check for NaNs after scaling
    if train_df[feature_cols].isnull().sum().sum() > 0:
        print("NaNs found in training features after scaling:")
        print(train_df[feature_cols].isnull().sum()[train_df[feature_cols].isnull().sum() > 0])
    if test_df[feature_cols].isnull().sum().sum() > 0:
        print("NaNs found in testing features after scaling:")
        print(test_df[feature_cols].isnull().sum()[test_df[feature_cols].isnull().sum() > 0])


    # Save scaler for later use
    joblib.dump(scaler, 'feature_scaler.pkl')
    joblib.dump(feature_cols, 'feature_columns.pkl')

    # Create sequences
    X_train, y_train, train_ids = create_sequences(train_df, feature_cols)
    X_test, y_test, test_ids = create_sequences(test_df, feature_cols)

    # For non-sequence models
    X_flat_train = X_train.reshape(X_train.shape[0], -1)
    X_flat_test = X_test.reshape(X_test.shape[0], -1)

    return (X_train, y_train, X_flat_train,
            X_test, y_test, X_flat_test, feature_cols)

In [ ]:
def build_lstm_model(input_shape):
    """Build LSTM model architecture"""
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=input_shape),
        Dropout(0.3),
        LSTM(64, return_sequences=True),
        Dropout(0.3),
        LSTM(32),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
    return model

In [ ]:
def build_gru_model(input_shape):
    """Build GRU model architecture"""
    model = Sequential([
        GRU(128, return_sequences=True, input_shape=input_shape),
        Dropout(0.3),
        GRU(64),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
    return model

In [ ]:
def build_transformer_model(input_shape, head_size=64, num_heads=4, ff_dim=4):
    """Build Transformer-based model"""
    inputs = Input(shape=input_shape)
    x = inputs

    # Normalization and Attention
    x = LayerNormalization(epsilon=1e-6)(x)
    x = MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=0.3
    )(x, x)
    x = LayerNormalization(epsilon=1e-6)(x)

    # Feed Forward Part
    x = Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(x)
    x = Dropout(0.3)(x)
    x = Conv1D(filters=input_shape[-1], kernel_size=1)(x)

    # Global Pooling and Output
    x = GlobalAveragePooling1D()(x)
    outputs = Dense(1)(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
    return model

In [ ]:
def build_cnn_model(input_shape):
    """Build 1D CNN model"""
    model = Sequential([
        Conv1D(64, kernel_size=3, activation='relu', input_shape=input_shape),
        MaxPooling1D(pool_size=2),
        Conv1D(128, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
    return model

In [ ]:
def train_models(X_train, y_train, X_test, y_test):
    """Train and evaluate multiple models"""
    models = {
        'LSTM': build_lstm_model((X_train.shape[1], X_train.shape[2])),
        'GRU': build_gru_model((X_train.shape[1], X_train.shape[2])),
        'Transformer': build_transformer_model((X_train.shape[1], X_train.shape[2])),
        'CNN': build_cnn_model((X_train.shape[1], X_train.shape[2]))
    }

    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)

    history = {}
    results = {}

    for name, model in models.items():
        print(f"\nTraining {name} model...")
        history[name] = model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=100,
            batch_size=64,
            callbacks=[early_stop, reduce_lr],
            verbose=1
        )

        # Evaluate on test set
        y_pred = model.predict(X_test).flatten()
        results[name] = {
            'model': model,
            'y_pred': y_pred,
            'y_true': y_test
        }

        # Save model
        model.save(f"{name.lower()}_rul_model.h5")

    return results, history

In [ ]:
# This code has been moved into the train_models function.
# early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
# reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)

# history = {}
# results = {}

# for name, model in models.items():
#     print(f"\nTraining {name} model...")
#     history[name] = model.fit(
#         X_train, y_train,
#         validation_split=0.2,
#         epochs=100,
#         batch_size=64,
#         callbacks=[early_stop, reduce_lr],
#         verbose=1
#     )

In [ ]:
def train_models(X_train, y_train, X_test, y_test):
    """Train and evaluate multiple models"""
    models = {
        'LSTM': build_lstm_model((X_train.shape[1], X_train.shape[2])),
        'GRU': build_gru_model((X_train.shape[1], X_train.shape[2])),
        'Transformer': build_transformer_model((X_train.shape[1], X_train.shape[2])),
        'CNN': build_cnn_model((X_train.shape[1], X_train.shape[2]))
    }

    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)

    history = {}
    results = {}

    for name, model in models.items():
        print(f"\nTraining {name} model...")
        history[name] = model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=100,
            batch_size=64,
            callbacks=[early_stop, reduce_lr],
            verbose=1
        )
        # Evaluate on test set
        y_pred = model.predict(X_test).flatten()
        results[name] = {
            'model': model,
            'y_pred': y_pred,
            'y_true': y_test
        }

        # Save model
        model.save(f"{name.lower()}_rul_model.h5")

    return results, history

In [ ]:
# 3. MODEL EVALUATION
# ----------------------------

def cmapss_score(y_true, y_pred):
    """NASA CMAPSS scoring function"""
    d = y_pred - y_true
    return np.mean(np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1))

def evaluate_results(results):
    """Evaluate model performance using multiple metrics"""
    evaluation = {}

    for name, res in results.items():
        y_true = res['y_true']
        y_pred = res['y_pred']

        evaluation[name] = {
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R2': r2_score(y_true, y_pred),
            'CMAPSS': cmapss_score(y_true, y_pred)
        }

    # Create comparison dataframe
    eval_df = pd.DataFrame(evaluation).T
    eval_df = eval_df.sort_values('CMAPSS')

    # Plot comparison
    plt.figure(figsize=(12, 8))
    eval_df[['MAE', 'RMSE']].plot(kind='bar', title='Error Metrics Comparison')
    plt.ylabel('Cycles')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('error_metrics.png')

    plt.figure(figsize=(10, 6))
    eval_df['CMAPSS'].plot(kind='bar', color='purple', title='CMAPSS Score Comparison')
    plt.ylabel('Score (lower is better)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('cmapss_score.png')

    return eval_df

In [ ]:
# MAIN EXECUTION
# ----------------------------

if __name__ == "__main__":
    # 1. Prepare data
    print("Preprocessing data...")
    train_path = "PM_train.csv"
    test_path = "PM_test.csv"
    (X_train, y_train, X_flat_train,
     X_test, y_test, X_flat_test, feature_cols) = prepare_data(train_path, test_path)

    print(f"Training sequences: {X_train.shape}")
    print(f"Testing sequences: {X_test.shape}")

    # 2. Train models
    print("\nTraining models...")
    results, history = train_models(X_train, y_train, X_test, y_test)

    # 3. Evaluate models
    print("\nEvaluating models...")
    eval_df = evaluate_results(results)
    print("\nModel Performance Comparison:")
    print(eval_df)

    # 4. Feature importance analysis
    print("\nAnalyzing feature importance...")
    importance_results = analyze_feature_importance(results, X_test, feature_cols)

    # 5. Save results
    eval_df.to_csv("model_performance.csv")
    for model_name, importance in importance_results.items():
        importance.to_csv(f"{model_name.lower()}_feature_importance.csv")

    print("\nAnalysis complete! Results saved to files.")

Preprocessing data...
Training sequences: (3563, 30, 17)
Testing sequences: (2063, 30, 17)

Training models...


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training LSTM model...
Epoch 1/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 2/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 3/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 4/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 5/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 6/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 2.0000e-04
Epoch 7/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 2.0000e-04
Epoch 8/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss


Training GRU model...
Epoch 1/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 2/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 3/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 4/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 5/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 6/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 2.0000e-04
Epoch 7/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 2.0000e-04
Epoch 8/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: nan 


Training Transformer model...
Epoch 1/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 10s 111ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 2/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 3/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 4/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 5/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 6/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 2.0000e-04
Epoch 7/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 2.0000e-04
Epoch 8/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - lo


Training CNN model...
Epoch 1/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 65ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 2/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 3/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 4/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 5/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 0.0010
Epoch 6/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 2.0000e-04
Epoch 7/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 2.0000e-04
Epoch 8/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: nan - 


Evaluating models...


ValueError: Input contains NaN.

In [ ]:
def cmapss_score(y_true, y_pred):
    d = y_pred - y_true
    return np.mean(np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1))

In [ ]:
def cmapss_score(y_true, y_pred):
    d = y_pred - y_true
    return np.mean(np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.ticker import MaxNLocator
import matplotlib.gridspec as gridspec

In [ ]:
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300

In [ ]:
def plot_sensor_trends(df, engine_ids=None, sensors=['s2', 's3', 's4', 's7', 's11', 's12']):
    """
    Plot sensor trends for selected engines with degradation visualization

    Args:
        df (DataFrame): Preprocessed dataset
        engine_ids (list): Specific engine IDs to plot (random if None)
        sensors (list): Sensors to visualize
    """
    if engine_ids is None:
        engine_ids = np.random.choice(df['id'].unique(), 3, replace=False)

    plt.figure(figsize=(14, 10))
    gs = gridspec.GridSpec(3, 2)

    # Create degradation timeline visualization
    ax0 = plt.subplot(gs[0, :])
    for engine_id in engine_ids:
        engine_data = df[df['id'] == engine_id]
        max_cycle = engine_data['cycle'].max()
        degradation = max_cycle - engine_data['cycle']
        ax0.plot(engine_data['cycle'], degradation, label=f'Engine {engine_id}', lw=2)

    ax0.set_title('Engine Degradation Timeline')
    ax0.set_xlabel('Operation Cycles')
    ax0.set_ylabel('Remaining Useful Life (RUL)')
    ax0.invert_yaxis()
    ax0.legend()
    ax0.grid(True, linestyle='--', alpha=0.7)

In [ ]:
def plot_correlation_matrix(df, sensors=None):
    """Plot sensor correlation matrix with degradation indicators"""
    if sensors is None:
        sensors = [f's{i}' for i in range(1, 22)] + ['RUL']

    plt.figure(figsize=(16, 14))
    corr = df[sensors].corr()

In [ ]:
def plot_correlation_matrix(df, sensors=None):
    """Plot sensor correlation matrix with degradation indicators"""
    if sensors is None:
        sensors = [f's{i}' for i in range(1, 22)] + ['RUL']

    plt.figure(figsize=(16, 14))
    corr = df[sensors].corr()

    # Create mask for upper triangle
    mask = np.triu(np.ones_like(corr, dtype=bool))

    # Create colormap with red for negative, blue for positive
    cmap = sns.diverging_palette(220, 20, as_cmap=True)

    # Plot heatmap
    ax = sns.heatmap(
        corr,
        mask=mask,
        cmap=cmap,
        center=0,
        annot=True,
        fmt=".2f",
        annot_kws={"size": 9},
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8}
    )

    # Highlight RUL correlations
    rul_corr = corr['RUL'].sort_values(ascending=False)
    for i, sensor in enumerate(rul_corr.index):
        if sensor != 'RUL':
            ax.add_patch(plt.Rectangle((i, corr.shape[0]-1), 1, 1, fill=False, edgecolor='red', lw=2))

    plt.title('Sensor Correlation Matrix with RUL', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('sensor_correlation.png', bbox_inches='tight')
    plt.close()

In [ ]:
# 2. MODEL TRAINING VISUALS
# ----------------------------

def plot_training_history(history, model_names):
    """Plot training and validation loss/MAE for multiple models"""
    plt.figure(figsize=(16, 10))

    for i, metric in enumerate(['loss', 'mae']):
        plt.subplot(2, 1, i+1)
        for model_name in model_names:
            if model_name in history:
                hist = history[model_name].history
                val_metric = f'val_{metric}'

                if metric in hist and val_metric in hist:
                    epochs = range(1, len(hist[metric]) + 1)

                    # Plot training metric
                    plt.plot(epochs, hist[metric], '--', alpha=0.8,
                             label=f'{model_name} Train')

                    # Plot validation metric
                    plt.plot(epochs, hist[val_metric], '-', linewidth=2,
                             label=f'{model_name} Validation')

        plt.title(f'Training vs Validation {metric.upper()}')
        plt.xlabel('Epochs')
        plt.ylabel(metric.upper())
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))

    plt.tight_layout()
    plt.savefig('training_history.png', bbox_inches='tight')
    plt.close()

In [ ]:
# 3. MODEL EVALUATION VISUALS
# ----------------------------

def plot_performance_comparison(eval_df):
    """Visualize model performance metrics"""
    plt.figure(figsize=(16, 12))

    # Create colormap
    colors = plt.cm.viridis(np.linspace(0, 1, len(eval_df)))

    # MAE and RMSE comparison
    plt.subplot(2, 2, 1)
    ax = eval_df[['MAE', 'RMSE']].plot(kind='bar', color=['skyblue', 'salmon'])
    plt.title('Error Metrics Comparison')
    plt.ylabel('Cycles')
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    # Add value labels
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}",
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 5),
                    textcoords='offset points')

    # R² comparison
    plt.subplot(2, 2, 2)
    plt.bar(eval_df.index, eval_df['R2'], color=colors)
    plt.title('R-squared Comparison')
    plt.ylabel('R² Score')
    plt.xticks(rotation=45)
    plt.ylim(0, 1)
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    # Add value labels
    for i, v in enumerate(eval_df['R2']):
        plt.text(i, v + 0.02, f"{v:.3f}", ha='center')

    # CMAPSS score comparison
    plt.subplot(2, 2, 3)
    plt.bar(eval_df.index, eval_df['CMAPSS'], color=colors)
    plt.title('CMAPSS Score Comparison')
    plt.ylabel('CMAPSS Score (lower is better)')
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    # Add value labels
    for i, v in enumerate(eval_df['CMAPSS']):
        plt.text(i, v + 0.1, f"{v:.1f}", ha='center')

    # Actual vs Predicted for best model
    best_model = eval_df['CMAPSS'].idxmin()
    plt.subplot(2, 2, 4)

    # Get predictions from results (assuming results dict available)
    if 'results' in globals():
        y_true = results[best_model]['y_true']
        y_pred = results[best_model]['y_pred']

        # Create hexbin plot for density visualization
        hb = plt.hexbin(y_true, y_pred, gridsize=50, cmap='Blues', mincnt=1)
        plt.colorbar(hb, label='Point Density')

        # Plot perfect prediction line
        max_val = max(y_true.max(), y_pred.max())
        plt.plot([0, max_val], [0, max_val], 'r--', alpha=0.7)

        plt.title(f'Actual vs Predicted RUL ({best_model})')
        plt.xlabel('Actual RUL (Cycles)')
        plt.ylabel('Predicted RUL (Cycles)')
        plt.grid(True, linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.savefig('performance_comparison.png', bbox_inches='tight')
    plt.close()

In [ ]:
# 4. MODEL INTERPRETABILITY VISUALS
# ----------------------------

def plot_feature_importance(importance_results, top_n=15):
    """Visualize feature importance across models"""
    plt.figure(figsize=(16, 12))

    # Create combined importance dataframe
    importance_df = pd.DataFrame(importance_results)

    # Normalize importance for each model
    importance_df = importance_df.apply(lambda x: x / x.abs().max(), axis=0)

    # Select top features across all models
    top_features = importance_df.max(axis=1).sort_values(ascending=False).head(top_n).index

    # Create subplots
    n_models = len(importance_df.columns)
    fig, axes = plt.subplots(n_models, 1, figsize=(14, 4*n_models))

    if n_models == 1:
        axes = [axes]

    for i, model_name in enumerate(importance_df.columns):
        # Get top features for this model
        model_importance = importance_df.loc[top_features, model_name].sort_values()

        # Create horizontal bar plot
        ax = axes[i]
        model_importance.plot(kind='barh', color='dodgerblue', ax=ax)

        # Add feature value labels
        for j, v in enumerate(model_importance.values):
            ax.text(v, j, f" {v:.3f}", va='center', fontsize=10)

        ax.set_title(f'{model_name} - Top Feature Importance', fontsize=14)
        ax.set_xlabel('Normalized Importance')
        ax.grid(axis='x', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig('feature_importance.png', bbox_inches='tight')
    plt.close()

In [ ]:
def plot_degradation_example(results, model_name, engine_id, X_test, test_ids, feature_cols, seq_length=30):
    """Plot actual vs predicted degradation for a specific engine"""
    # Get engine data
    engine_indices = np.where(test_ids == engine_id)[0]
    if len(engine_indices) == 0:
        print(f"Engine {engine_id} not found in test data")
        return

    # Get predictions
    engine_preds = results[model_name]['y_pred'][engine_indices]
    engine_true = results[model_name]['y_true'][engine_indices]

    # Get sensor data
    sensor_idx = 4  # Example sensor index
    sensor_name = feature_cols[sensor_idx]
    sensor_data = X_test[engine_indices, -1, sensor_idx]  # Last cycle in each sequence

    # Create timeline
    cycles = np.arange(len(engine_indices)) * 5 + seq_length

    plt.figure(figsize=(14, 8))

    # Plot RUL predictions
    plt.subplot(2, 1, 1)
    plt.plot(cycles, engine_true, 'b-', linewidth=2, label='Actual RUL')
    plt.plot(cycles, engine_preds, 'r--', linewidth=2, label='Predicted RUL')
    plt.fill_between(cycles, engine_true, engine_preds, where=(engine_preds < engine_true),
                     color='red', alpha=0.2, label='Early Warning')
    plt.fill_between(cycles, engine_true, engine_preds, where=(engine_preds >= engine_true),
                     color='green', alpha=0.2, label='Conservative Estimate')

    plt.title(f'Engine {engine_id} Degradation Profile - {model_name}', fontsize=16)
    plt.xlabel('Operation Cycles')
    plt.ylabel('Remaining Useful Life')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)

    # Plot key sensor
    plt.subplot(2, 1, 2)
    plt.plot(cycles, sensor_data, 'g-', linewidth=2, label=f'Sensor {sensor_name}')
    plt.xlabel('Operation Cycles')
    plt.ylabel('Sensor Value')
    plt.title(f'Key Degradation Indicator: {sensor_name}', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(f'degradation_profile_engine_{engine_id}.png', bbox_inches='tight')
    plt.close()


In [ ]:
def plot_degradation_example(results, model_name, engine_id, X_test, test_ids, feature_cols, seq_length=30):
    """Plot actual vs predicted degradation for a specific engine"""
    # Get engine data
    engine_indices = np.where(test_ids == engine_id)[0]
    if len(engine_indices) == 0:
        print(f"Engine {engine_id} not found in test data")
        return

    # Get predictions
    engine_preds = results[model_name]['y_pred'][engine_indices]
    engine_true = results[model_name]['y_true'][engine_indices]

    # Get sensor data
    sensor_idx = 4  # Example sensor index
    sensor_name = feature_cols[sensor_idx]
    sensor_data = X_test[engine_indices, -1, sensor_idx]  # Last cycle in each sequence

    # Create timeline
    cycles = np.arange(len(engine_indices)) * 5 + seq_length

    plt.figure(figsize=(14, 8))

    # Plot RUL predictions
    plt.subplot(2, 1, 1)
    plt.plot(cycles, engine_true, 'b-', linewidth=2, label='Actual RUL')
    plt.plot(cycles, engine_preds, 'r--', linewidth=2, label='Predicted RUL')
    plt.fill_between(cycles, engine_true, engine_preds, where=(engine_preds < engine_true),
                     color='red', alpha=0.2, label='Early Warning')
    plt.fill_between(cycles, engine_true, engine_preds, where=(engine_preds >= engine_true),
                     color='green', alpha=0.2, label='Conservative Estimate')

    plt.title(f'Engine {engine_id} Degradation Profile - {model_name}', fontsize=16)
    plt.xlabel('Operation Cycles')
    plt.ylabel('Remaining Useful Life')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)

    # Plot key sensor
    plt.subplot(2, 1, 2)
    plt.plot(cycles, sensor_data, 'g-', linewidth=2, label=f'Sensor {sensor_name}')
    plt.xlabel('Operation Cycles')
    plt.ylabel('Sensor Value')
    plt.title(f'Key Degradation Indicator: {sensor_name}', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(f'degradation_profile_engine_{engine_id}.png', bbox_inches='tight')
    plt.close()

In [ ]:
def generate_all_visuals(df, history, results, eval_df, importance_results,
                         X_test=None, test_ids=None, feature_cols=None):
    """Generate all visualizations in a comprehensive report"""
    print("Generating data exploration visuals...")
    plot_sensor_trends(df)
    plot_correlation_matrix(df)

    print("Generating model training visuals...")
    plot_training_history(history, list(history.keys()))

    print("Generating model evaluation visuals...")
    plot_performance_comparison(eval_df)

    print("Generating interpretability visuals...")
    plot_feature_importance(importance_results)

    if X_test is not None and test_ids is not None and feature_cols is not None:
        print("Generating engine degradation profiles...")
        example_engines = np.random.choice(np.unique(test_ids), 3, replace=False)
        best_model = eval_df['CMAPSS'].idxmin()

        for engine_id in example_engines:
            plot_degradation_example(
                results, best_model, engine_id, X_test, test_ids, feature_cols
            )

    print("Visualization report complete! All figures saved.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import pandas as pd

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")

In [ ]:
def visualize_model_architectures():
    """Create schematic diagrams of model architectures"""
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle("Model Architecture Comparison", fontsize=22, fontweight='bold')

    # LSTM Architecture
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.set_title("LSTM Architecture", fontsize=16, fontweight='bold')
    ax1.text(0.5, 0.8, "Input Sequence", ha='center', fontsize=14,
             bbox=dict(facecolor='lightblue', alpha=0.5))
    ax1.text(0.5, 0.6, "LSTM Layer (128 units)", ha='center', fontsize=14,
             bbox=dict(facecolor='lightgreen', alpha=0.5))
    ax1.text(0.5, 0.4, "LSTM Layer (64 units)", ha='center', fontsize=14,
             bbox=dict(facecolor='lightgreen', alpha=0.5))
    ax1.text(0.5, 0.2, "Dense Layer (32 units)", ha='center', fontsize=14,
             bbox=dict(facecolor='salmon', alpha=0.5))
    ax1.text(0.5, 0.0, "RUL Output", ha='center', fontsize=14,
             bbox=dict(facecolor='gold', alpha=0.5))
    ax1.axis('off')

    # GRU Architecture
    ax2 = fig.add_subplot(2, 2, 2)
    ax2.set_title("GRU Architecture", fontsize=16, fontweight='bold')
    ax2.text(0.5, 0.8, "Input Sequence", ha='center', fontsize=14,
             bbox=dict(facecolor='lightblue', alpha=0.5))
    ax2.text(0.5, 0.6, "GRU Layer (128 units)", ha='center', fontsize=14,
             bbox=dict(facecolor='lightgreen', alpha=0.5))
    ax2.text(0.5, 0.4, "GRU Layer (64 units)", ha='center', fontsize=14,
             bbox=dict(facecolor='lightgreen', alpha=0.5))
    ax2.text(0.5, 0.2, "Dense Layer (32 units)", ha='center', fontsize=14,
             bbox=dict(facecolor='salmon', alpha=0.5))
    ax2.text(0.5, 0.0, "RUL Output", ha='center', fontsize=14,
             bbox=dict(facecolor='gold', alpha=0.5))
    ax2.axis('off')

    # Transformer Architecture
    ax3 = fig.add_subplot(2, 2, 3)
    ax3.set_title("Transformer Architecture", fontsize=16, fontweight='bold')
    ax3.text(0.5, 0.9, "Input Sequence", ha='center', fontsize=14,
             bbox=dict(facecolor='lightblue', alpha=0.5))
    ax3.text(0.5, 0.7, "Positional Encoding", ha='center', fontsize=14,
             bbox=dict(facecolor='lightpink', alpha=0.5))
    ax3.text(0.5, 0.5, "Multi-Head Attention (4 heads)", ha='center', fontsize=14,
             bbox=dict(facecolor='lightgreen', alpha=0.5))
    ax3.text(0.5, 0.3, "Feed Forward Network", ha='center', fontsize=14,
             bbox=dict(facecolor='salmon', alpha=0.5))
    ax3.text(0.5, 0.1, "RUL Output", ha='center', fontsize=14,
             bbox=dict(facecolor='gold', alpha=0.5))
    ax3.axis('off')

    # CNN Architecture
    ax4 = fig.add_subplot(2, 2, 4)
    ax4.set_title("CNN Architecture", fontsize=16, fontweight='bold')
    ax4.text(0.5, 0.8, "Input Sequence", ha='center', fontsize=14,
             bbox=dict(facecolor='lightblue', alpha=0.5))
    ax4.text(0.5, 0.6, "Conv1D (64 filters)", ha='center', fontsize=14,
             bbox=dict(facecolor='lightgreen', alpha=0.5))
    ax4.text(0.5, 0.4, "Conv1D (128 filters)", ha='center', fontsize=14,
             bbox=dict(facecolor='lightgreen', alpha=0.5))
    ax4.text(0.5, 0.2, "Flatten", ha='center', fontsize=14,
             bbox=dict(facecolor='salmon', alpha=0.5))
    ax4.text(0.5, 0.0, "RUL Output", ha='center', fontsize=14,
             bbox=dict(facecolor='gold', alpha=0.5))
    ax4.axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('model_architectures.png', dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# ----------------------------
# 2. MODEL PERFORMANCE VISUALS
# ----------------------------

def create_performance_dashboard(eval_df):
    """Create interactive dashboard of model performance metrics"""
    fig = make_subplots(
        rows=2, cols=2,
        specs=[[{"type": "bar"}, {"type": "bar"}],
               [{"type": "scatter"}, {"type": "bar"}]],
        subplot_titles=("MAE & RMSE Comparison", "R² Score Comparison",
                        "Actual vs Predicted RUL", "CMAPSS Score Comparison")
    )

    # MAE and RMSE Comparison
    fig.add_trace(
        go.Bar(
            x=eval_df.index,
            y=eval_df['MAE'],
            name='MAE',
            marker_color='#1f77b4',
            text=[f"{v:.1f}" for v in eval_df['MAE']],
            textposition='outside'
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(
            x=eval_df.index,
            y=eval_df['RMSE'],
            name='RMSE',
            marker_color='#ff7f0e',
            text=[f"{v:.1f}" for v in eval_df['RMSE']],
            textposition='outside'
        ),
        row=1, col=1
    )

    # R² Comparison
    fig.add_trace(
        go.Bar(
            x=eval_df.index,
            y=eval_df['R2'],
            name='R²',
            marker_color='#2ca02c',
            text=[f"{v:.3f}" for v in eval_df['R2']],
            textposition='outside'
        ),
        row=1, col=2
    )

    # Actual vs Predicted (Best model)
    best_model = eval_df['CMAPSS'].idxmin()
    y_true = results[best_model]['y_true']
    y_pred = results[best_model]['y_pred']

    fig.add_trace(
        go.Scatter(
            x=y_true,
            y=y_pred,
            mode='markers',
            marker=dict(
                size=8,
                color=np.abs(y_true - y_pred),
                colorscale='Viridis',
                showscale=True,
                colorbar=dict(title='Error (Cycles)')
            ),
            name='Predictions',
            text=[f"True: {t}, Pred: {p}" for t, p in zip(y_true, y_pred)]
        ),
        row=2, col=1
    )

    # Add perfect prediction line
    max_val = max(y_true.max(), y_pred.max())
    fig.add_trace(
        go.Scatter(
            x=[0, max_val],
            y=[0, max_val],
            mode='lines',
            line=dict(color='red', dash='dash'),
            name='Perfect Prediction'
        ),
        row=2, col=1
    )

    # CMAPSS Score Comparison
    fig.add_trace(
        go.Bar(
            x=eval_df.index,
            y=eval_df['CMAPSS'],
            name='CMAPSS',
            marker_color='#d62728',
            text=[f"{v:.1f}" for v in eval_df['CMAPSS']],
            textposition='outside'
        ),
        row=2, col=2
    )

    # Update layout
    fig.update_layout(
        title=f'Model Performance Dashboard (Best: {best_model})',
        height=900,
        showlegend=False,
        hovermode='closest'
    )

    # Axis labels
    fig.update_xaxes(title_text="Models", row=1, col=1)
    fig.update_yaxes(title_text="Cycles", row=1, col=1)
    fig.update_xaxes(title_text="Models", row=1, col=2)
    fig.update_yaxes(title_text="R² Score", range=[0, 1.1], row=1, col=2)
    fig.update_xaxes(title_text="Actual RUL (Cycles)", row=2, col=1)
    fig.update_yaxes(title_text="Predicted RUL (Cycles)", row=2, col=1)
    fig.update_xaxes(title_text="Models", row=2, col=2)
    fig.update_yaxes(title_text="CMAPSS Score", row=2, col=2)

    fig.write_html("performance_dashboard.html")
    fig.write_image("performance_dashboard.png", width=1400, height=900)

    return fig

In [ ]:
# ----------------------------
# 3. ERROR ANALYSIS VISUALS
# ----------------------------

def plot_error_distribution(results):
    """Plot error distribution across models"""
    plt.figure(figsize=(14, 8))

    # Prepare error data
    error_data = []
    for model_name, res in results.items():
        errors = res['y_pred'] - res['y_true']
        error_data.append(pd.DataFrame({
            'Model': model_name,
            'Error': errors
        }))

    error_df = pd.concat(error_data)

    # Create violin plot
    sns.violinplot(
        x='Model',
        y='Error',
        data=error_df,
        inner='quartile',
        palette='viridis',
        cut=0
    )

    # Add horizontal lines and annotations
    plt.axhline(0, color='red', linestyle='--', alpha=0.7)
    plt.text(0.5, 5, 'Early Fail Prediction (Conservative)',
             fontsize=12, color='red', ha='center')
    plt.text(0.5, -5, 'Late Fail Prediction (Dangerous)',
             fontsize=12, color='red', ha='center')

    plt.title('Error Distribution Across Models', fontsize=18)
    plt.xlabel('Models', fontsize=14)
    plt.ylabel('Prediction Error (Cycles)', fontsize=14)
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig('error_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()

    return error_df

In [ ]:
# 4. INTERPRETABILITY VISUALS
# ----------------------------

def plot_global_feature_importance(importance_results):
    """Plot global feature importance across models"""
    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(2, 2)

    # Create subplots for each model
    for i, (model_name, importance) in enumerate(importance_results.items()):
        ax = fig.add_subplot(gs[i//2, i%2])

        # Get top 10 features
        top_features = importance.head(10)

        # Plot horizontal bar chart
        y_pos = np.arange(len(top_features))
        ax.barh(y_pos, top_features.values, align='center', color='dodgerblue')
        ax.set_yticks(y_pos)
        ax.set_yticklabels(top_features.index)
        ax.invert_yaxis()
        ax.set_title(f'{model_name} - Feature Importance', fontsize=14)
        ax.set_xlabel('Importance Score', fontsize=12)

        # Add value labels
        for j, v in enumerate(top_features.values):
            ax.text(v, j, f" {v:.3f}", va='center', fontsize=10)

    plt.suptitle('Global Feature Importance Comparison', fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('global_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
def plot_temporal_importance(explanation_data, feature_cols, model_name):
    """Plot temporal feature importance for sequence models"""
    # Create time-indexed importance matrix
    time_steps = explanation_data.shape[1]
    importance_matrix = np.abs(explanation_data).mean(0)

    # Create heatmap
    plt.figure(figsize=(16, 10))
    sns.heatmap(
        importance_matrix.T,
        cmap="viridis",
        yticklabels=feature_cols,
        xticklabels=[f"t-{time_steps-i}" for i in range(time_steps)],
        cbar_kws={'label': 'Importance Score'}
    )

    plt.title(f'Temporal Feature Importance - {model_name}', fontsize=16)
    plt.xlabel('Time Steps Before Failure', fontsize=14)
    plt.ylabel('Sensor Features', fontsize=14)

    plt.tight_layout()
    plt.savefig(f'temporal_importance_{model_name.lower()}.png', dpi=300, bbox_inches='tight')
    plt.close()

    return importance_matrix

In [ ]:
def plot_engine_degradation(engine_id, true_rul, pred_rul, sensor_data, model_name):
    """Plot degradation profile for a specific engine"""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

    # Plot RUL predictions
    cycles = np.arange(len(true_rul))
    ax1.plot(cycles, true_rul, 'b-', linewidth=2.5, label='Actual RUL')
    ax1.plot(cycles, pred_rul, 'r--', linewidth=2.5, label=f'{model_name} Prediction')

    # Fill between for early/late warnings
    ax1.fill_between(cycles, true_rul, pred_rul,
                     where=(pred_rul < true_rul),
                     color='red', alpha=0.2, label='Early Warning')
    ax1.fill_between(cycles, true_rul, pred_rul,
                     where=(pred_rul >= true_rul),
                     color='green', alpha=0.2, label='Conservative Estimate')

    ax1.set_title(f'Engine {engine_id} Degradation Profile', fontsize=18)
    ax1.set_ylabel('Remaining Useful Life (Cycles)', fontsize=14)
    ax1.legend(loc='upper right')
    ax1.grid(True, linestyle='--', alpha=0.7)

    # Plot key sensors
    for sensor_name, values in sensor_data.items():
        ax2.plot(cycles, values, label=sensor_name, linewidth=2)

    ax2.set_title('Critical Sensor Trends', fontsize=16)
    ax2.set_xlabel('Operation Cycles', fontsize=14)
    ax2.set_ylabel('Normalized Sensor Value', fontsize=14)
    ax2.legend(loc='upper right')
    ax2.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(f'degradation_engine_{engine_id}.png', dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# ----------------------------
# 6. INTERACTIVE MODEL COMPARISON
# ----------------------------

def create_interactive_comparison(eval_df, error_df):
    """Create interactive model comparison dashboard"""
    # Prepare data
    plot_df = eval_df.reset_index().rename(columns={'index': 'Model'})
    plot_df = plot_df.melt(id_vars='Model',
                           value_vars=['MAE', 'RMSE', 'R2', 'CMAPSS'],
                           var_name='Metric',
                           value_name='Value')

    # Create parallel coordinates plot
    fig = px.parallel_coordinates(
        eval_df.reset_index(),
        color='CMAPSS',
        dimensions=['MAE', 'RMSE', 'R2', 'CMAPSS'],
        labels={'MAE': 'MAE (Cycles)',
                'RMSE': 'RMSE (Cycles)',
                'R2': 'R² Score',
                'CMAPSS': 'CMAPSS Score'},
        color_continuous_scale=px.colors.diverging.Tealrose,
        title='Model Performance Comparison'
    )

    fig.update_layout(
        height=600,
        coloraxis_colorbar=dict(title='CMAPSS Score',
                                orientation='h',
                                y=-0.2,
                                thickness=15)
    )

    # Create error distribution plot
    fig2 = px.box(
        error_df,
        x='Model',
        y='Error',
        color='Model',
        points="all",
        hover_data=['Error'],
        title='Prediction Error Distribution'
    )

    fig2.update_layout(
        height=500,
        showlegend=False,
        yaxis_title="Prediction Error (Cycles)"
    )

    # Combine into dashboard
    dashboard = make_subplots(
        rows=2, cols=1,
        specs=[[{"type": "parcoords"}], [{"type": "box"}]],
        subplot_titles=("Model Performance Comparison", "Error Distribution")
    )

    dashboard.add_trace(fig.data[0], row=1, col=1)
    for trace in fig2.data:
        dashboard.add_trace(trace, row=2, col=1)

    dashboard.update_layout(
        title_text="Interactive Model Comparison Dashboard",
        height=1000,
        showlegend=False
    )

    dashboard.write_html("interactive_comparison.html")

    return dashboard

In [ ]:
# ----------------------------
# MAIN VISUALIZATION EXECUTION
# ----------------------------

def generate_all_visualizations(eval_df, results, importance_results,
                                explanation_data, feature_cols):
    """Generate all visualizations in a comprehensive report"""
    print("Generating model architecture visuals...")
    visualize_model_architectures()

    print("Creating performance dashboard...")
    create_performance_dashboard(eval_df)

    print("Analyzing error distribution...")
    error_df = plot_error_distribution(results)

    print("Visualizing feature importance...")
    plot_global_feature_importance(importance_results)

    print("Generating temporal importance plots...")
    for model_name in ['LSTM', 'GRU', 'Transformer']:
        if model_name in explanation_data:
            plot_temporal_importance(explanation_data[model_name], feature_cols, model_name)

    print("Creating interactive comparison...")
    create_interactive_comparison(eval_df, error_df)

    print("Visualization report complete! All figures saved.")

    # Return paths to created visualizations
    return {
        'architectures': 'model_architectures.png',
        'dashboard': 'performance_dashboard.html',
        'error_distribution': 'error_distribution.png',
        'feature_importance': 'global_feature_importance.png',
        'temporal_importance': [f'temporal_importance_{m.lower()}.png' for m in ['LSTM', 'GRU', 'Transformer']],
        'interactive': 'interactive_comparison.html'
    }

# Example usage:
# visualizations = generate_all_visualizations(eval_df, results, importance_results,
#                                             explanation_data, feature_cols)